In [ ]:
# ============================================================
# 03_training_evaluation.ipynb  —  ATLAS GO Prediction Pipeline
# Trains  : MLP, CNN1D, TransformerGO on 9 feature combinations
# Metrics : TRUE CAFA protein-centric Fmax + macro-AUPRC
#           Per-protein pr/rc averaged then maximised over thresholds
#           Matches CAFA3 / CAFA4 official evaluation protocol
# Outputs : 27-model results + publication CSV/LaTeX tables
# Run after: 02_feature_extraction.ipynb
# ============================================================


In [1]:
## ── 1. Mount ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
## ── 2. Imports + Config ──────────────────────────────────────
import yaml, os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import average_precision_score
from tqdm import tqdm
import warnings; warnings.filterwarnings('ignore')

cfg_path = "/content/drive/MyDrive/atlas-go-dynamicsV2/configs/config.yaml"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

data_processed = cfg['paths']['data_processed']
results_dir    = cfg['paths']['results']
SEED           = cfg['random_seed']

os.makedirs(results_dir, exist_ok=True)
os.makedirs(f"{data_processed}/models", exist_ok=True)

# Reproducibility
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

g = torch.Generator(); g.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | Seed: {SEED}')


Device: cuda | Seed: 42


In [3]:
## ── 3. Load Labels + Features ───────────────────────────────
Y_all  = torch.FloatTensor(np.load(f"{data_processed}/Y_all.npy"))
splits = np.load(f"{data_processed}/splits.npz")
train_idx = splits['train_idx']
val_idx   = splits['val_idx']
test_idx  = splits['test_idx']

# Dynamic ontology slices from 01_data_preparation
with open(f"{data_processed}/go_ontology_slices.yaml") as f:
    ont = yaml.safe_load(f)
N_CLASSES = Y_all.shape[1]
MFO_slice = slice(ont['MFO']['start'], ont['MFO']['end'])
BPO_slice = slice(ont['BPO']['start'], ont['BPO']['end'])
CCO_slice = slice(ont['CCO']['start'], ont['CCO']['end'])

print(f"Y_all: {Y_all.shape} | N_CLASSES={N_CLASSES}")
print(f"MFO={ont['MFO']['end']-ont['MFO']['start']} BPO={ont['BPO']['end']-ont['BPO']['start']} CCO={ont['CCO']['end']-ont['CCO']['start']}")
print(f"Splits: train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")

FEATURES = ['p','ps','psd','e','es','esd','pe','pes','pesd']
X_dict = {}
for fname in FEATURES:
    X = np.load(f"{data_processed}/X_{fname}.npy")
    X_dict[fname] = torch.FloatTensor(X)
    print(f"  X_{fname}: {X.shape}")
print('Features loaded.')


Y_all: torch.Size([1289, 495]) | N_CLASSES=495
MFO=156 BPO=220 CCO=119
Splits: train=1031 val=129 test=129
  X_p: (1289, 1024)
  X_ps: (1289, 1028)
  X_psd: (1289, 1032)
  X_e: (1289, 1280)
  X_es: (1289, 1284)
  X_esd: (1289, 1288)
  X_pe: (1289, 2304)
  X_pes: (1289, 2308)
  X_pesd: (1289, 2312)
Features loaded.


In [4]:
## ── 4. TRUE CAFA Protein-centric Fmax ───────────────────────
#
# CAFA standard (Zhou et al. 2019; CAFA3/CAFA4 evaluation):
#   For threshold t:
#     pr_i(t) = |P_i(t) ∩ T_i| / |P_i(t)|   [protein-level precision]
#     rc_i(t) = |P_i(t) ∩ T_i| / |T_i|       [protein-level recall]
#   AvgPr(t) = mean over proteins with >=1 prediction
#   AvgRc(t) = mean over ALL proteins
#   F(t)     = 2 * AvgPr(t) * AvgRc(t) / (AvgPr(t) + AvgRc(t))
#   Fmax     = max_t F(t)
#
# NOTE: This differs from sklearn micro/macro f1_score.

def cafa_fmax(y_proba: np.ndarray, y_true: np.ndarray,
              thresholds=None) -> float:
    if thresholds is None:
        thresholds = np.arange(0.01, 1.00, 0.01)
    best_f = 0.0
    for t in thresholds:
        y_pred      = (y_proba >= t).astype(np.float32)
        pred_cnt    = y_pred.sum(axis=1)            # (N,)
        true_cnt    = y_true.sum(axis=1)            # (N,)
        tp          = (y_pred * y_true).sum(axis=1) # (N,)
        has_pred    = pred_cnt > 0
        if has_pred.sum() == 0:
            continue
        pr_per      = np.where(has_pred, tp / np.maximum(pred_cnt, 1e-9), 0.0)
        avg_pr      = pr_per[has_pred].mean()
        rc_per      = np.where(true_cnt > 0, tp / np.maximum(true_cnt, 1e-9), 0.0)
        avg_rc      = rc_per.mean()
        if avg_pr + avg_rc < 1e-9:
            continue
        f = 2 * avg_pr * avg_rc / (avg_pr + avg_rc)
        if f > best_f:
            best_f = f
    return float(best_f)


def compute_full_metrics(y_proba: torch.Tensor, y_true: torch.Tensor) -> dict:
    """CAFA Fmax + macro-AUPRC for Overall / MFO / BPO / CCO."""
    yp = y_proba.numpy(); yt = y_true.numpy()
    results = {}
    for name, sl in [('Overall',slice(None)),('MFO',MFO_slice),('BPO',BPO_slice),('CCO',CCO_slice)]:
        yp_s = yp[:, sl]; yt_s = yt[:, sl]
        fmax  = cafa_fmax(yp_s, yt_s)
        valid = yt_s.sum(axis=0) > 0
        auprc = float(average_precision_score(yt_s[:,valid], yp_s[:,valid], average='macro')) if valid.sum() > 0 else 0.0
        results[f'{name}_Fmax']  = round(fmax,  3)
        results[f'{name}_AUPRC'] = round(auprc, 3)
    return results

# Sanity check
_p = np.array([[0.9,0.1,0.8],[0.2,0.7,0.3]])
_t = np.array([[1,  0,  1  ],[0,  1,  0  ]])
assert cafa_fmax(_p, _t) > 0.9, 'CAFA Fmax sanity check failed'
print(f'CAFA Fmax sanity check passed: {cafa_fmax(_p,_t):.3f}')
print('Metrics functions ready.')


CAFA Fmax sanity check passed: 1.000
Metrics functions ready.


In [5]:
## ── 5. Naive Baseline ────────────────────────────────────────
print('Computing Naive baseline (top-10% train frequency)...')
train_freq   = Y_all[train_idx].mean(dim=0)
thresh_naive = torch.quantile(train_freq, 0.90)
Y_naive      = (train_freq > thresh_naive).float().unsqueeze(0).repeat(len(test_idx),1)
naive_metrics = compute_full_metrics(Y_naive, Y_all[test_idx])
print(f"  Overall : Fmax={naive_metrics['Overall_Fmax']:.3f}  AUPRC={naive_metrics['Overall_AUPRC']:.3f}")
print(f"  MFO={naive_metrics['MFO_Fmax']:.3f}  BPO={naive_metrics['BPO_Fmax']:.3f}  CCO={naive_metrics['CCO_Fmax']:.3f}")


Computing Naive baseline (top-10% train frequency)...
  Overall : Fmax=0.092  AUPRC=0.018
  MFO=0.108  BPO=0.034  CCO=0.112


In [6]:
## ── 6. Model Architectures ───────────────────────────────────

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__(); self.alpha=alpha; self.gamma=gamma
    def forward(self, logits, targets):
        bce  = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt   = torch.exp(-bce)
        return (self.alpha*(1-pt)**self.gamma*bce).mean()


class MLP(nn.Module):
    """input → 1024 → 512 → 256 → num_classes (BN + Dropout)."""
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(1024,512),       nn.BatchNorm1d(512),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512,256),                              nn.ReLU(),
            nn.Linear(256,num_classes),
        )
    def forward(self, x): return self.net(x)


class CNN1D(nn.Module):
    """1-D CNN: feature vector treated as single-channel signal."""
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.c1=nn.Conv1d(1,128,7,padding=3);   self.b1=nn.BatchNorm1d(128)
        self.c2=nn.Conv1d(128,64,5,padding=2);  self.b2=nn.BatchNorm1d(64)
        self.c3=nn.Conv1d(64,32,3,padding=1);   self.b3=nn.BatchNorm1d(32)
        self.drop=nn.Dropout(0.3)
        flat=self._flat(input_dim)
        self.fc1=nn.Linear(flat,512); self.out=nn.Linear(512,num_classes)
    def _flat(self,d):
        with torch.no_grad():
            x=torch.zeros(1,1,d)
            x=F.relu(self.b1(self.c1(x))); x=F.relu(self.b2(self.c2(x))); x=F.relu(self.b3(self.c3(x)))
            return x.view(1,-1).size(1)
    def forward(self,x):
        x=x.unsqueeze(1)
        x=F.relu(self.b1(self.c1(x))); x=F.relu(self.b2(self.c2(x))); x=F.relu(self.b3(self.c3(x)))
        return self.out(self.drop(F.relu(self.fc1(x.view(x.size(0),-1)))))


class TransformerGO(nn.Module):
    """Residual feed-forward Transformer for multi-label GO prediction."""
    def __init__(self, input_dim, num_classes, d_model=512, num_layers=3):
        super().__init__()
        self.embed  = nn.Linear(input_dim, d_model)
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, d_model*2), nn.GELU(), nn.Dropout(0.1),
                nn.Linear(d_model*2, d_model), nn.Dropout(0.1),
            ) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)
    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers: x = x + layer(x)  # residual
        return self.head(self.norm(x))


print('Architectures defined: MLP | CNN1D | TransformerGO')


Architectures defined: MLP | CNN1D | TransformerGO


In [7]:
## ── 7. Generic Train + Evaluate ─────────────────────────────

def train_model(ModelClass, feat, bs, lr, wd, max_ep=100, patience=10, sfx=''):
    """
    Train on feat, early-stop on val CAFA Fmax, save best weights.
    Returns best val Fmax.
    """
    X       = X_dict[feat]
    loader  = DataLoader(TensorDataset(X[train_idx], Y_all[train_idx]),
                         batch_size=bs, shuffle=True,
                         num_workers=2, worker_init_fn=seed_worker, generator=g)
    model   = ModelClass(X.shape[1], N_CLASSES).to(device)
    opt     = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    crit    = FocalLoss(cfg['training']['focal_loss_alpha'], cfg['training']['focal_loss_gamma'])
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg['training']['cosine_annealing_tmax'])
    save_p  = f"{data_processed}/models/{feat}{sfx}.pth"
    best_f  = 0.0; pat = 0
    X_val   = X[val_idx].to(device); Y_val = Y_all[val_idx]

    for ep in range(max_ep):
        model.train()
        for Xb,Yb in loader:
            Xb,Yb = Xb.to(device),Yb.to(device)
            opt.zero_grad()
            loss = crit(model(Xb),Yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['training']['gradient_clip_norm'])
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            vp = torch.sigmoid(model(X_val)).cpu()
        vf = cafa_fmax(vp.numpy(), Y_val.numpy())
        if vf > best_f:
            best_f=vf; pat=0; torch.save(model.state_dict(), save_p)
        else:
            pat+=1
        if ep%20==0: print(f'    ep{ep:3d}: val_Fmax={vf:.4f} best={best_f:.4f} pat={pat}/{patience}')
        if pat>=patience: print(f'    Early stop ep{ep}'); break

    del model; torch.cuda.empty_cache()
    return best_f


def evaluate_model(ModelClass, feat, sfx='') -> dict:
    X = X_dict[feat]
    m = ModelClass(X.shape[1], N_CLASSES).to(device)
    m.load_state_dict(torch.load(f"{data_processed}/models/{feat}{sfx}.pth", map_location=device))
    m.eval()
    with torch.no_grad():
        p = torch.sigmoid(m(X[test_idx].to(device))).cpu()
    del m; torch.cuda.empty_cache()
    return compute_full_metrics(p, Y_all[test_idx])


print('train_model / evaluate_model ready.')


train_model / evaluate_model ready.


In [8]:
## ── 8. Train + Evaluate MLP ──────────────────────────────────
print('='*55+'\n MLP (9 features)\n'+'='*55)
mlp_rows = []
for feat in tqdm(FEATURES, desc='MLP'):
    print(f'\n  [{feat.upper()}] dim={X_dict[feat].shape[1]}')
    train_model(MLP, feat, cfg['training']['mlp_batch_size'],
                cfg['training']['mlp_learning_rate'], cfg['training']['mlp_weight_decay'], sfx='_mlp')
    m = evaluate_model(MLP, feat, '_mlp')
    m['Feature']='  '+feat.upper(); m['Architecture']='MLP'; mlp_rows.append(m)
    print(f"  Test Fmax={m['Overall_Fmax']:.3f} MFO={m['MFO_Fmax']:.3f} BPO={m['BPO_Fmax']:.3f} CCO={m['CCO_Fmax']:.3f}")
df_mlp = pd.DataFrame(mlp_rows)
df_mlp['Feature'] = df_mlp['Feature'].str.strip()
df_mlp.to_csv(f"{results_dir}/MLP_Results.csv", index=False)
print(f"\nMLP DONE — BEST: {df_mlp.loc[df_mlp['Overall_Fmax'].idxmax(),'Feature']} Fmax={df_mlp['Overall_Fmax'].max():.3f}")


 MLP (9 features)


MLP:   0%|          | 0/9 [00:00<?, ?it/s]


  [P] dim=1024
    ep  0: val_Fmax=0.0316 best=0.0316 pat=0/10
    ep 20: val_Fmax=0.2801 best=0.2851 pat=3/10
    ep 40: val_Fmax=0.2970 best=0.2999 pat=1/10
    ep 60: val_Fmax=0.3046 best=0.3084 pat=7/10
    ep 80: val_Fmax=0.3080 best=0.3164 pat=9/10
    Early stop ep81


MLP:  11%|█         | 1/9 [00:21<02:52, 21.60s/it]

  Test Fmax=0.294 MFO=0.266 BPO=0.100 CCO=0.337

  [PS] dim=1028
    ep  0: val_Fmax=0.0224 best=0.0224 pat=0/10
    ep 20: val_Fmax=0.2786 best=0.2870 pat=1/10
    ep 40: val_Fmax=0.2993 best=0.3075 pat=2/10
    Early stop ep48


...

  Test Fmax=0.312 MFO=0.286 BPO=0.116 CCO=0.354

  [PES] dim=2308
    ep  0: val_Fmax=0.0301 best=0.0301 pat=0/10
    ep 20: val_Fmax=0.3109 best=0.3251 pat=3/10
    Early stop ep27


MLP:  89%|████████▉ | 8/9 [01:15<00:07,  7.92s/it]

  Test Fmax=0.286 MFO=0.249 BPO=0.094 CCO=0.323

  [PESD] dim=2312
    ep  0: val_Fmax=0.0230 best=0.0230 pat=0/10
    ep 20: val_Fmax=0.3061 best=0.3098 pat=2/10
    Early stop ep39


MLP: 100%|██████████| 9/9 [01:22<00:00,  9.22s/it]

  Test Fmax=0.304 MFO=0.256 BPO=0.109 CCO=0.346

MLP DONE — BEST: ESD Fmax=0.312


In [9]:
## ── 9. Train + Evaluate CNN1D ────────────────────────────────
print('='*55+'\n CNN1D (9 features)\n'+'='*55)
cnn_rows = []
for feat in tqdm(FEATURES, desc='CNN1D'):
    print(f'\n  [{feat.upper()}] dim={X_dict[feat].shape[1]}')
    train_model(CNN1D, feat, cfg['training']['cnn_batch_size'],
                cfg['training']['cnn_learning_rate'], cfg['training']['cnn_weight_decay'], sfx='_cnn')
    m = evaluate_model(CNN1D, feat, '_cnn')
    m['Feature']=feat.upper(); m['Architecture']='CNN1D'; cnn_rows.append(m)
    print(f"  Test Fmax={m['Overall_Fmax']:.3f} MFO={m['MFO_Fmax']:.3f} BPO={m['BPO_Fmax']:.3f} CCO={m['CCO_Fmax']:.3f}")
df_cnn = pd.DataFrame(cnn_rows)
df_cnn.to_csv(f"{results_dir}/CNN1D_Results.csv", index=False)
print(f"\nCNN1D DONE — BEST: {df_cnn.loc[df_cnn['Overall_Fmax'].idxmax(),'Feature']} Fmax={df_cnn['Overall_Fmax'].max():.3f}")


 CNN1D (9 features)


CNN1D:   0%|          | 0/9 [00:00<?, ?it/s]


  [P] dim=1024
    ep  0: val_Fmax=0.0285 best=0.0285 pat=0/10
    ep 20: val_Fmax=0.2909 best=0.3019 pat=1/10
    ep 40: val_Fmax=0.3339 best=0.3367 pat=2/10
    ep 60: val_Fmax=0.3409 best=0.3409 pat=0/10
    ep 80: val_Fmax=0.3502 best=0.3513 pat=9/10
    Early stop ep94


CNN1D:  11%|█         | 1/9 [00:45<06:00, 45.02s/it]

  Test Fmax=0.308 MFO=0.271 BPO=0.079 CCO=0.365

  [PS] dim=1028
    ep  0: val_Fmax=0.0586 best=0.0586 pat=0/10
    ep 20: val_Fmax=0.3004 best=0.3004 pat=0/10
    ep 40: val_Fmax=0.3271 best=0.3434 pat=10/10
    Early stop ep40


...

CNN1D:  89%|████████▉ | 8/9 [04:59<00:43, 43.89s/it]

  Test Fmax=0.327 MFO=0.282 BPO=0.114 CCO=0.383

  [PESD] dim=2312
    ep  0: val_Fmax=0.0304 best=0.0304 pat=0/10
    ep 20: val_Fmax=0.3078 best=0.3108 pat=4/10
    ep 40: val_Fmax=0.3536 best=0.3555 pat=1/10
    ep 60: val_Fmax=0.3673 best=0.3673 pat=0/10
    Early stop ep77


CNN1D: 100%|██████████| 9/9 [06:19<00:00, 42.13s/it]

  Test Fmax=0.334 MFO=0.315 BPO=0.128 CCO=0.385



CNN1D DONE — BEST: E Fmax=0.338


In [10]:
## ── 10. Train + Evaluate TransformerGO ──────────────────────
print('='*55+'\n TransformerGO (9 features)\n'+'='*55)
trans_rows = []
for feat in tqdm(FEATURES, desc='Transformer'):
    print(f'\n  [{feat.upper()}] dim={X_dict[feat].shape[1]}')
    train_model(TransformerGO, feat, cfg['training']['transformer_batch_size'],
                cfg['training']['transformer_learning_rate'], cfg['training']['transformer_weight_decay'], sfx='_transformer')
    m = evaluate_model(TransformerGO, feat, '_transformer')
    m['Feature']=feat.upper(); m['Architecture']='Transformer'; trans_rows.append(m)
    print(f"  Test Fmax={m['Overall_Fmax']:.3f} MFO={m['MFO_Fmax']:.3f} BPO={m['BPO_Fmax']:.3f} CCO={m['CCO_Fmax']:.3f}")
df_transformer = pd.DataFrame(trans_rows)
df_transformer.to_csv(f"{results_dir}/Transformer_Results.csv", index=False)
print(f"\nTransformer DONE — BEST: {df_transformer.loc[df_transformer['Overall_Fmax'].idxmax(),'Feature']} Fmax={df_transformer['Overall_Fmax'].max():.3f}")


 TransformerGO (9 features)


Transformer:   0%|          | 0/9 [00:00<?, ?it/s]


  [P] dim=1024
    ep  0: val_Fmax=0.0441 best=0.0441 pat=0/10
    ep 20: val_Fmax=0.2784 best=0.2814 pat=4/10
    ep 40: val_Fmax=0.3237 best=0.3259 pat=4/10
    Early stop ep54


Transformer:  11%|█         | 1/9 [00:12<01:42, 12.85s/it]

  Test Fmax=0.306 MFO=0.288 BPO=0.115 CCO=0.341

  [PS] dim=1028
    ep  0: val_Fmax=0.0297 best=0.0297 pat=0/10
    ep 20: val_Fmax=0.2666 best=0.2825 pat=4/10
    ep 40: val_Fmax=0.3141 best=0.3141 pat=0/10
    Early stop ep55


...

  Test Fmax=0.329 MFO=0.297 BPO=0.107 CCO=0.365

  [PES] dim=2308
    ep  0: val_Fmax=0.0675 best=0.0675 pat=0/10
    ep 20: val_Fmax=0.3411 best=0.3411 pat=0/10
    ep 40: val_Fmax=0.3593 best=0.3614 pat=2/10
    ep 60: val_Fmax=0.3674 best=0.3682 pat=3/10
    ep 80: val_Fmax=0.3787 best=0.3901 pat=2/10
    Early stop ep97


Transformer:  89%|████████▉ | 8/9 [01:50<00:15, 15.95s/it]

  Test Fmax=0.343 MFO=0.318 BPO=0.154 CCO=0.355

  [PESD] dim=2312
    ep  0: val_Fmax=0.0498 best=0.0498 pat=0/10
    ep 20: val_Fmax=0.3257 best=0.3257 pat=0/10
    ep 40: val_Fmax=0.3578 best=0.3670 pat=2/10
    Early stop ep53


Transformer: 100%|██████████| 9/9 [02:03<00:00, 13.69s/it]

  Test Fmax=0.333 MFO=0.307 BPO=0.130 CCO=0.347

Transformer DONE — BEST: PES Fmax=0.343


In [11]:
## ── 11. Publication Tables ────────────────────────────────────
df_mlp         = pd.read_csv(f"{results_dir}/MLP_Results.csv")
df_cnn         = pd.read_csv(f"{results_dir}/CNN1D_Results.csv")
df_transformer = pd.read_csv(f"{results_dir}/Transformer_Results.csv")
df_mlp['Architecture']='MLP'; df_cnn['Architecture']='CNN1D'; df_transformer['Architecture']='Transformer'
all_res = pd.concat([df_mlp, df_cnn, df_transformer], ignore_index=True)

COLS = ['Architecture','Feature','Overall_Fmax','Overall_AUPRC','MFO_Fmax','BPO_Fmax','CCO_Fmax']

# TABLE A: Full 27-model ablation
tA = all_res[COLS].sort_values(['Architecture','Overall_Fmax'],ascending=[True,False]).round(3).reset_index(drop=True)
tA.to_csv(f"{results_dir}/TableA_Full27_Ablation.csv", index=False)
tA.to_latex(f"{results_dir}/TableA_Full27_Ablation.tex", index=False)
print('TABLE A — Full 27-Model Ablation (CAFA Fmax)')
print(tA.to_string(index=False))

# TABLE B: Best per architecture vs Naive
naive_row = {'Architecture':'Naive','Feature':'N/A',
             'Overall_Fmax':naive_metrics['Overall_Fmax'],
             'Overall_AUPRC':naive_metrics['Overall_AUPRC'],
             'MFO_Fmax':naive_metrics['MFO_Fmax'],
             'BPO_Fmax':naive_metrics['BPO_Fmax'],
             'CCO_Fmax':naive_metrics['CCO_Fmax']}
def brow(df,arch):
    r=df.loc[df['Overall_Fmax'].idxmax()]
    return {c:r[c] for c in COLS if c!='Architecture'}|{'Architecture':arch}
tB = pd.DataFrame([naive_row, brow(df_mlp,'MLP'), brow(df_cnn,'CNN1D'), brow(df_transformer,'Transformer')]).round(3)
tB.to_csv(f"{results_dir}/TableB_Best_vs_Naive.csv", index=False)
tB.to_latex(f"{results_dir}/TableB_Best_vs_Naive.tex", index=False)
print('\nTABLE B — Best Architecture vs Naive')
print(tB.to_string(index=False))

# Improvement
nf = naive_metrics['Overall_Fmax']
print(f'\nImprovement over Naive (Fmax={nf:.3f}):')
for arch,df_a in [('MLP',df_mlp),('CNN1D',df_cnn),('Transformer',df_transformer)]:
    f=df_a['Overall_Fmax'].max()
    print(f'  {arch:<12}: {f:.3f}  (+{(f-nf)/nf*100:.0f}%)')


TABLE A — Full 27-Model Ablation (CAFA Fmax)
Architecture Feature  Overall_Fmax  Overall_AUPRC  MFO_Fmax  BPO_Fmax  CCO_Fmax
       CNN1D       E         0.338          0.233     0.300     0.110     0.362
       CNN1D     ESD         0.336          0.248     0.304     0.116     0.363
       CNN1D    PESD         0.334          0.234     0.315     0.128     0.385
       CNN1D     PES         0.327          0.230     0.282     0.114     0.383
       CNN1D      PE         0.325          0.186     0.277     0.114     0.373
       CNN1D      ES         0.323          0.220     0.291     0.091     0.355
       CNN1D     PSD         0.318          0.198     0.290     0.109     0.339
       CNN1D       P         0.308          0.233     0.271     0.079     0.365
       CNN1D      PS         0.294          0.194     0.264     0.113     0.339
         MLP     ESD         0.312          0.207     0.257     0.106     0.337
         MLP      PE         0.312          0.223     0.286     0.116     0

In [12]:
## ── 12. ESM2 vs ATLAS Summary ────────────────────────────────
print('='*55)
print(' KEY FINDING: ESM2 vs ATLAS-only Features')
print('='*55)
esm2_feats=['E','ES','ESD']; atlas_feats=['P','PS','PSD']
for arch,df_a in [('MLP',df_mlp),('CNN1D',df_cnn),('Transformer',df_transformer)]:
    e=df_a[df_a['Feature'].isin(esm2_feats)]['Overall_Fmax'].mean()
    a=df_a[df_a['Feature'].isin(atlas_feats)]['Overall_Fmax'].mean()
    print(f'  {arch:<12}: ESM2_avg={e:.3f}  ATLAS_avg={a:.3f}  ESM2 +{(e-a)/a*100:.0f}% over ATLAS-only')
print(f'\nAll tables saved to: {results_dir}')
print('Next → run 04_xai_figures.ipynb')


 KEY FINDING: ESM2 vs ATLAS-only Features
  MLP         : ESM2_avg=0.307  ATLAS_avg=0.283  ESM2 +8% over ATLAS-only
  CNN1D       : ESM2_avg=0.332  ATLAS_avg=0.307  ESM2 +8% over ATLAS-only
  Transformer : ESM2_avg=0.331  ATLAS_avg=0.303  ESM2 +9% over ATLAS-only

All tables saved to: /content/drive/MyDrive/atlas-go-dynamicsV2/results/tables
Next → run 04_xai_figures.ipynb
